# Phase 2 — EEG Dataset Preparation

I convert labeled recordings into numerical epochs and audit their provenance and integrity.

> **Research integrity:** Figures labeled synthetic are not clinical EEG observations. Run code locally in VS Code using the project `.venv` interpreter.

## 1. Load a small simulated test dataset

The demonstration data are intentionally synthetic and allow inspection without any download.

In [ ]:
from pathlib import Path
import numpy as np
from eeg_sleep.synthetic import make_demo_dataset
from eeg_sleep.dataset import load_arrays
p=Path("data/processed/synthetic_demo.npz")
if not p.exists(): make_demo_dataset(p,n_subjects=10,repeats_per_stage=2)
X,y,groups,ids,fs=load_arrays(p)
print("Shape:",X.shape,"Labels:",np.unique(y,return_counts=True),"Hz:",fs)

## 2. Subject grouping and class distribution

In [ ]:
from eeg_sleep.constants import STAGES
from eeg_sleep.splits import subject_split
tr,va,te=subject_split(groups,seed=42)
for name,idx in [("train",tr),("val",va),("test",te)]:
    print(name,len(idx),"epochs",len(set(groups[idx])),"subjects")
    print("counts:",dict(zip(STAGES,np.bincount(y[idx],minlength=5))))
assert not set(groups[tr]).intersection(groups[te])

## 3. Visualize one epoch — the synthetic source is not clinical data

In [ ]:
import matplotlib.pyplot as plt
t=np.arange(X.shape[-1])/fs
fig,ax=plt.subplots(2,1,sharex=True,figsize=(10,4))
for c in range(2): ax[c].plot(t,X[0,c],linewidth=.5); ax[c].set_ylabel(f"Channel {c+1} (µV)")
ax[-1].set_xlabel("Time (s)");fig.suptitle("SIMULATED epoch")
plt.tight_layout();plt.show()

## 4. Real Sleep-EDF preparation

Use the CLI in the integrated VS Code terminal. These steps are **not** run automatically from the notebook:

```bash
python scripts/run_pipeline.py fetch --subjects 0 1 2 3 4 --recordings 1
python scripts/run_pipeline.py prepare
```

`dataset.py` reads EDF annotations, maps labels, filters continuous EEG and stores epochs together with subject IDs.

In [ ]:
real=Path("data/processed/sleep_edf.npz")
if real.exists():
    X_real,y_real,groups_real,ids_real,fs_real=load_arrays(real)
    print("REAL data:",X_real.shape,"Subjects:",len(set(groups_real)))
else: print("Real EEG has not been downloaded; please run fetch/prepare when ready.")

## Exercises

1. Describe why `(N,2,3000)` is the correct shape. 2. Explain how original R&K labels are converted to five classes. 3. Explain why `--max-epochs-per-record` can bias a dataset. 4. Why must subject metadata be saved with each epoch?